# VIX Term Structure

- Contango (正向市场)：意味着远期波动率预期高于近月
  - 正向市场 - literally "forward market"
  - 期货升水 - "futures premium"
  - 远期升水 - "forward premium"
- Backwardation (反向市场)：近月波动率高于远月
  - 反向市场 - literally "reverse market"
  - 期货贴水 - "futures discount"
  - 远期贴水 - "forward discount"

## Configuration
[top](#contents)

In [ ]:
from pydantic import BaseModel
from decimal import Decimal


class VixTermTechnical(BaseModel):
    name: str
    trading_day: str
    interval: str
    high: Decimal
    low: Decimal
    close: Decimal
    change_percent: Decimal
    change_text: str
    volume: Decimal
    volume_text: str
    volume_interpretation: str
    median_1year_volume: Decimal
    supertrend_bias: str
    support_level: Decimal
    support_date: str
    resistance_level: Decimal
    resistance_date: str

## Market Data
[top](#contents)

In [ ]:
import yfinance as yf
import pandas as pd


def get_vix_term_data() -> pd.DataFrame:
    # Define the ticker symbol for the S&P 500 index
    # tickers = ["^VIX9D", "^VIX", "^VIX3M", "^VIX6M", "^VIX1Y"]
    tickers = ["^VIX9D", "^VIX", "^VIX3M", "^VIX6M"]

    # Download historical data
    df = yf.download(tickers, period="5d", interval="1d", auto_adjust=True)
    df = df.drop(["High", "Low", "Open", "Volume"], axis=1)
    df = df.dropna()

    columns = df.columns.get_level_values(1).tolist()
    columns = [s.replace("^", "") for s in columns]

    # Flatten multi-index columns
    if isinstance(df.columns, pd.MultiIndex):
        column_names = ["Date"] + columns
        df = df.reset_index()
        df.columns = column_names
        df = df.set_index("Date")

    # Round to two decimal
    df[columns] = df[columns].round(2)

    df = df[[s.replace("^", "") for s in tickers]]

    # Keep the last row only
    df_last_row = df.iloc[-1:]

    # Save to CSV
    df_last_row.to_csv("../output/data/vix_term/prices.csv")
    print("Data saved to data/vix_term/prices.csv")

    return df_last_row

In [ ]:
df = get_vix_term_data()
df

In [ ]:
date = df.index[0]
print(date.strftime("%Y-%m-%d"))

In [ ]:
df_t = df.transpose()
df_t.columns = ["Value"]
#df_t["Day"] = [9, 30, 90, 180, 360]
df_t["Day"] = [9, 30, 90, 180]
df_t

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import make_interp_spline
from scipy.interpolate import interp1d

fonts = [
    "/usr/share/fonts/opentype/noto/NotoSerifCJK-Regular.ttc",
    "/usr/local/share/fonts/演示春风楷.ttf",
    "/usr/local/share/fonts/演示佛系体.ttf",
    "/usr/local/share/fonts/演示秋鸿楷.ttf",
    "/usr/local/share/fonts/演示夏行楷.ttf",
    "/usr/local/share/fonts/演示悠然小楷.ttf",
]
font_path = fonts[0]  # the location of the font file
cn_font = fm.FontProperties(fname=font_path)  # get the font based on the font_path

fig = plt.figure(figsize=(16, 9))
plt.scatter(df_t["Day"], df_t["Value"])
plt.plot(
    df_t["Day"],
    df_t["Value"],
    # ylim=(0, 50),
    color="black",
    # marker="o",
    # mfc="red",
    # mec="red",
)

plt.xlabel("到期日", fontproperties=cn_font)
plt.ylabel("数值", fontproperties=cn_font)
plt.title("恐慌指数期限结构", fontproperties=cn_font)

# ax.spines["bottom"].set_color("gray")
# ax.spines["top"].set_color("gray")
# ax.spines["left"].set_color("gray")
# ax.spines["right"].set_color("gray")

plt.grid(linestyle="--", linewidth=0.5, alpha=0.5)
plt.axhline(40, color="black", linestyle="--", label="Extreme", linewidth=0.5)
plt.axhline(30, color="maroon", linestyle="--", label="Very High", linewidth=0.5)
plt.axhline(25, color="red", linestyle="--", label="High", linewidth=0.5)
plt.axhline(20, color="orange", linestyle="--", label="Mod High", linewidth=0.5)
plt.axhline(16, color="lime", linestyle="--", label="Normal", linewidth=0.5)

plt.colorbar()

plt.savefig(
    "../output/data/vix_term/chart.svg",
    dpi=240,
    transparent=False,
    bbox_inches="tight",
    pad_inches=0.1,
)
plt.show()

In [ ]:
import numpy as np


def analyze_term_structure(vix_series):

    front = vix_series.loc["VIX"]["Value"]
    back = vix_series.loc["VIX3M"]["Value"]

    # Check structure
    if front < back:
        structure = "Contango"
    elif front > back:
        structure = "Backwardation"
    else:
        structure = "Flat"

    # Calculate pairwise differences
    diffs = vix_series["Value"].diff().dropna()

    # Check structure
    if all(diffs > 0):
        is_mixed = False
    elif all(diffs < 0):
        is_mixed = False
    else:
        is_mixed = True

    # Select two specific points by index (example: first and last)
    x1, y1 = vix_series.loc["VIX"][["Day", "Value"]]
    x2, y2 = vix_series.loc["VIX3M"][["Day", "Value"]]

    # Calculate slope: (y2 - y1) / (x2 - x1)
    slope = (y2 - y1) / (x2 - x1)

    return {
        "structure": structure,
        "is_mixed": is_mixed,
        "pairwise_diffs": diffs.to_dict(),
        "slope": slope,
    }

In [ ]:
result = analyze_term_structure(df_t)
print(f"Structure: {result['structure']}")
print(f"Is Mixed: {result['is_mixed']}")
d = result["pairwise_diffs"]
for i in d:
    print(f"{i}: {d[i]:.2f}")
print(f"Slope: {result['slope']:.4f}")

## Video Script

In [ ]:
def get_asymmetric_slope_desc_en(slope: float) -> str:
    if slope > 0.6:
        return "a steep upward curve — typical of strong contango"
    elif slope > 0.2:
        return "a modestly rising term structure"
    elif slope > 0.05:
        return "a slightly upward-sloping curve"
    elif -0.05 <= slope <= 0.05:
        return "a flat or neutral curve"
    elif slope < -0.3:
        return "a sharply inverted curve — signaling backwardation"
    else:  # -0.3 <= slope < -0.05
        return "a slightly inverted curve"

In [ ]:
def get_asymmetric_slope_desc_zh(slope: float) -> str:
    if slope > 0.6:  # Contango
        return "陡峭上升的曲线走势"
    elif slope > 0.2:
        return "温和上升的期限结构走势"
    elif slope > 0.05:
        return "略微上升的曲线走势"
    elif -0.05 <= slope <= 0.05:
        return "平坦或中性的曲线走势"
    elif slope < -0.3:  # Backwardation
        return "明显倒挂的曲线走势"
    else:  # -0.3 <= slope < -0.05
        return "轻微倒挂的曲线走势"

In [ ]:
def vix_term_structure_script_en(
    structure: str, is_mixed: bool, slope: float, num_months: int
):
    slope_desc = get_asymmetric_slope_desc(slope)

    intro = "Now let’s take a look at the VIX term structure."
    slope_text = f"The curve shows {slope_desc} across the front {num_months} months."

    if structure == "Contango":
        interpretation = (
            "This indicates a typical contango setup, where longer-dated volatility expectations "
            "remain higher than near-term fear. It's often a sign that the market is relatively calm "
            "and that risk is perceived to be lower in the short term."
        )
    elif structure == "Backwardation":
        interpretation = (
            "We’re seeing a backwardation pattern, meaning near-term volatility is priced higher "
            "than the longer-term. This usually reflects elevated short-term fear, often due to event risk "
            "or market stress."
        )
    else:
        interpretation = (
            "The term structure is currently mixed, with no consistent upward or downward curve. "
            "This reflects uncertainty — possibly a transition between market regimes or a lack of consensus "
            "about forward risk."
        )

    # closing = "We’ll keep watching the curve closely for any shifts that might signal a change in market sentiment."
    closing = ""

    return "\n".join([intro, slope_text, interpretation, closing])

In [ ]:
def vix_term_structure_script_zh(
    structure: str, is_mixed: bool, slope: float, num_months: int
):
    slope_desc = get_asymmetric_slope_desc_zh(slope)

    intro = "接下来我们来看一下'恐慌指数期限结构'的变化。"
    slope_text = f"目前曲线在前{num_months}个月呈现{slope_desc}。"

    if structure == "Contango":
        interpretation = (
            "这显示出典型的'正向市场'结构，意味着远期波动率预期高于近月，"
            "通常反映市场情绪平稳，短期内对风险的担忧较低。"
        )
    elif structure == "Backwardation":
        interpretation = (
            "目前市场呈现'反向市场'结构，近月波动率高于远月，"
            "这往往反映市场在短期内存在较强的不确定性或风险事件。"
        )
    else:
        interpretation = (
            "当前期限结构表现为混合状态，曲线未明显上升或下降，"
            "这反映出市场的不确定性，可能是市场阶段转换的过渡期，"
            "或者投资者对未来风险尚未形成一致预期。"
        )

    closing = ""

    return "\n".join([intro, slope_text, interpretation, closing])

In [ ]:
from pulse.utils.script import save_script

# Example usage:
result = analyze_term_structure(df_t)

structure = result["structure"]
is_mixed = result["is_mixed"]
slope = result["slope"]
num_months = len(df_t)

script = vix_term_structure_script_zh(
    structure=structure, is_mixed=is_mixed, slope=slope, num_months=num_months
)

filename = "../output/data/vix_term/script.txt"
save_script(script, filename)

print(script)